In [4]:
import pandas as pd

In [5]:
movies = pd.read_csv("../datasets/ml-latest/movies.csv")
ratings = pd.read_csv("../datasets/ml-latest/ratings.csv")
tags = pd.read_csv("../datasets/ml-latest/tags.csv")

In [6]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86537 entries, 0 to 86536
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  86537 non-null  int64 
 1   title    86537 non-null  object
 2   genres   86537 non-null  object
dtypes: int64(1), object(2)
memory usage: 2.0+ MB


In [7]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33832162 entries, 0 to 33832161
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 1.0 GB


In [8]:
tags.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2328315 entries, 0 to 2328314
Data columns (total 4 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   userId     int64 
 1   movieId    int64 
 2   tag        object
 3   timestamp  int64 
dtypes: int64(3), object(1)
memory usage: 71.1+ MB


In [9]:
tags.drop(columns=["userId"], inplace=True)

In [10]:
# Replace | with space in movies.genres and make it lowercase
movies["genres"] = movies["genres"].str.replace("|", " ").str.lower().fillna("").astype(str)

In [11]:
# Group all tags for the same movieId into one text string in tags DataFrame
tags = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(str(s) for s in x)).str.lower().reset_index()
tags.rename(columns={"tag": "all_tags"}, inplace=True)

In [ ]:
# Merge movies with grouped tags
movies = movies.merge(tags, on="movieId", how="left")
movies["all_tags"] = movies["all_tags"].fillna("").astype(str)

In [13]:
# Create final text column
movies["combined_features"] = movies["genres"] + " " + movies["all_tags"]

In [14]:
# Remove extra spaces
movies["combined_features"] = movies["combined_features"].str.replace(r"\s+", " ", regex=True).str.strip()

In [15]:
movies

,movieId,title,genres,all_tags,combined_features
0,1,Toy Story (1995),adventure animation children comedy fantasy,animation friendship toys animation disney pix...,adventure animation children comedy fantasy an...
1,2,Jumanji (1995),adventure children fantasy,animals based on a book fantasy magic board ga...,adventure children fantasy animals based on a ...
2,3,Grumpier Old Men (1995),comedy romance,sequel moldy old old age old men wedding old p...,comedy romance sequel moldy old old age old me...
3,4,Waiting to Exhale (1995),comedy drama romance,characters chick flick girl movie characters c...,comedy drama romance characters chick flick gi...
4,5,Father of the Bride Part II (1995),comedy,family pregnancy wedding 4th wall aging baby d...,comedy family pregnancy wedding 4th wall aging...
...,...,...,...,...,...
86532,288967,State of Siege: Temple Attack (2021),action drama,,action drama
86533,288971,Ouija Japan (2021),action horror,,action horror
86534,288975,The Men Who Made the Movies: Howard Hawks (1973),documentary,,documentary
86535,288977,Skinford: Death Sentence (2023),crime thriller,,crime thriller


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(movies["combined_features"])